In [1]:
import os
import pickle
import torch
from torch_geometric.data import HeteroData
import random

PROCESSED_DIR = "Data"

def build_graph_from_combos(
    max_examples=2000,      # limit number of examples to process
    max_edges_per_drug=20,  # cap number of edges per drug
    seed=42
):
    random.seed(seed)
    torch.manual_seed(seed)

    # Load combos
    with open(os.path.join(PROCESSED_DIR, "combos.pkl"), "rb") as f:
        ds = pickle.load(f)

    examples = ds["examples"]
    se_ids = ds["se_ids"]

    # Limit how many examples to process
    examples = examples[:max_examples]

    num_drugs = max([max(ex["drug_idxs"]) for ex in examples]) + 1
    num_sidefx = len(se_ids)

    print(f"Building graph with {num_drugs} drugs, {num_sidefx} side effects...")
    print(f"Processing up to {len(examples)} examples.")

    data = HeteroData()
    data["drug"].x = torch.randn(num_drugs, 128)
    data["sideeffect"].x = torch.randn(num_sidefx, 64)

    drug_edges, se_edges = [], []

    for ex in examples:
        drugs = ex["drug_idxs"]
        labels = ex["label"]
        pos_sidefx = [i for i, l in enumerate(labels) if l == 1]

        # Limit per drug edges to keep the graph manageable
        if len(pos_sidefx) > max_edges_per_drug:
            pos_sidefx = random.sample(pos_sidefx, max_edges_per_drug)

        for d in drugs:
            for s in pos_sidefx:
                drug_edges.append(d)
                se_edges.append(s)

    total_edges = len(drug_edges)
    print(f"Total edges created: {total_edges}")

    if total_edges == 0:
        raise ValueError("No edges created! Check your 'label' data in combos.pkl")

    # Assign edges
    edge_index = torch.tensor([drug_edges, se_edges], dtype=torch.long)
    data["drug", "causes", "sideeffect"].edge_index = edge_index
    data["sideeffect", "caused_by", "drug"].edge_index = edge_index.flip(0)

    save_path = os.path.join(PROCESSED_DIR, f"hoddi_v2_graph_limited.pt")
    torch.save(data, save_path)
    print(f"✅ Saved new graph to {save_path}")

if __name__ == "__main__":
    build_graph_from_combos(
        max_examples=2000,      # you can set 100, 1000, etc.
        max_edges_per_drug=15,  # lower = lighter
        seed=42
    )


Building graph with 768 drugs, 1000 side effects...
Processing up to 2000 examples.
Total edges created: 57904
✅ Saved new graph to Data\hoddi_v2_graph_limited.pt
